# Thermal model

The node network in Python, and how it was fitted. No board.

In [1]:
SIMULATED = True          # False, and PORT, at the bench
PORT = 'COM4'

`coaxial.thermal` carries the same network the firmware runs: ten nodes, driver and phase per leg, mcu, regulators, afe, and the board to ambient. Only `board_to_ambient` and `board_capacity` have a clean measurement behind them.

In [2]:
from coaxial import thermal

print('ambient %.1f C' % thermal.AMBIENT)
print('board_to_ambient %.2f K/W, board_capacity %.0f J/K, tau %.1f min'
      % (thermal.CFG['board_to_ambient'], thermal.CFG['board_capacity'],
         thermal.tau_minutes()))
for node in thermal.NODES:
    print('%-12s to_board %5.1f K/W  capacity %.3f J/K'
          % (node, thermal.CFG['to_board'][node], thermal.CFG['capacity'][node]))

ambient 20.0 C
board_to_ambient 8.33 K/W, board_capacity 49 J/K, tau 6.8 min
driver_u     to_board  45.6 K/W  capacity 0.117 J/K
driver_v     to_board  45.6 K/W  capacity 0.117 J/K
driver_w     to_board  45.6 K/W  capacity 0.117 J/K
phase_u      to_board  45.6 K/W  capacity 0.400 J/K
phase_v      to_board  45.6 K/W  capacity 0.400 J/K
phase_w      to_board  45.6 K/W  capacity 0.400 J/K
mcu          to_board  22.5 K/W  capacity 0.900 J/K
regulators   to_board  15.0 K/W  capacity 0.800 J/K
afe          to_board  41.5 K/W  capacity 0.300 J/K


The NTC sits in the drivers' hot spot: an offset over the board taken in the passive state, and a coupling to the drivers' rise solved from the switching state.

In [3]:
print(thermal.MEASURED)
print('NTC_OFFSET %.2f K' % thermal.NTC_OFFSET)
print('NTC_SEES_DRIVERS %.3f' % thermal.NTC_SEES_DRIVERS)
print('driver rise while switching %.1f K' % thermal.DRIVER_RISE_SWITCHING)
for state in thermal.STATES:
    print('%-8s %s' % (state, thermal.STATE_IS[state]))

{'passive': {'ntc': 36.0, 'board': 30.0}, 'switching': {'ntc': 55.6, 'board': 40.0}}
NTC_OFFSET 6.00 K
NTC_SEES_DRIVERS 1.055
driver rise while switching 9.1 K
passive  AFE off: the drivers have supply, no PWM
afe      AFE on: drivers unpowered, sensors alive, no traffic
traffic  AFE on: DAQ at full tilt, data off the board
switch   AFE off: three legs at 50 %


In [4]:
steady = thermal.steady(thermal.POWER_SWITCHING)
print('power while switching: %.2f W' % sum(thermal.POWER_SWITCHING.values()))
for node in thermal.ALL_NODES:
    print('%-12s %6.2f C' % (node, steady[node]))
print('NTC expected %.2f C' % thermal.expected_ntc(steady['board'], steady['driver_v'] - steady['board']))
for minutes in (5, 10, 25):
    print('%2d min: %.0f %% of the way to equilibrium' % (minutes, 100 * thermal.settled_fraction(minutes)))

power while switching: 2.40 W
driver_u      49.11 C
driver_v      49.11 C
driver_w      49.11 C
phase_u       39.99 C
phase_v       39.99 C
phase_w       39.99 C
mcu           54.98 C
regulators    57.00 C
afe           39.99 C
board         39.99 C
NTC expected 55.61 C
 5 min: 52 % of the way to equilibrium
10 min: 77 % of the way to equilibrium
25 min: 97 % of the way to equilibrium


`calibrate` is the fit: `to_board = (T_zone - T_reference) / P_zone`, the camera's surface temperature at each source against a reference patch of soldermask at the same moment.

In [5]:
camera = {'mcu': 40.0 + 20.0, 'regulators': 40.0 + 10.1,
          'driver_u': 40.0 + 17.3, 'driver_v': 40.0 + 17.3, 'driver_w': 40.0 + 17.3}
fit = thermal.calibrate(camera, board_c=40.0)
for node, k_per_w in fit.items():
    print('%-12s %6.1f K/W  (model %5.1f)' % (node, k_per_w, thermal.CFG['to_board'][node]))

mcu            30.0 K/W  (model  22.5)
regulators      8.9 K/W  (model  15.0)
driver_u       86.5 K/W  (model  45.6)
driver_v       86.5 K/W  (model  45.6)
driver_w       86.5 K/W  (model  45.6)


In [6]:
from coaxial import thermalmap

print(thermalmap.render({n: steady[n] for n in thermal.NODES}, steady['board'],
                        cells=60, colour=False, title='steady state, switching'))

  steady state, switching

                                                cccccccccccccccccccccccc                                                  @@ 100 C
                                        cccccccccccccccccccccccccccccccccccccccc                                          WW
                                  cccccccccccccccccccccccccccccccccccccccccccccccccccc                                    WW
                              cccccccccccccccccccccccccccccccccccccccccccccccccccccccccccc                                88
                          cccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccc                            88
                        cccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccc                          88
                    cccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccc                      %%
                  cccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccc

## Conclusions

In [7]:
measured = ('board_to_ambient', 'board_capacity')
print('measured against the supply and the camera:')
for name in measured:
    print('   %-18s %.2f' % (name, thermal.CFG[name]))
print('solved from the camera states:')
print('   %-18s %.2f K' % ('NTC_OFFSET', thermal.NTC_OFFSET))
print('   %-18s %.3f' % ('NTC_SEES_DRIVERS', thermal.NTC_SEES_DRIVERS))
print('per leg, three times the lumped zone the camera saw:')
for node in thermal.DRIVERS + thermal.PHASES:
    print('   %-18s %.1f K/W' % (node, thermal.CFG['to_board'][node]))
print('fitted here against the same states:')
for node, k_per_w in sorted(fit.items()):
    print('   %-18s %6.1f K/W against the model %5.1f'
          % (node, k_per_w, thermal.CFG['to_board'][node]))

measured against the supply and the camera:
   board_to_ambient   8.33
   board_capacity     49.00
solved from the camera states:
   NTC_OFFSET         6.00 K
   NTC_SEES_DRIVERS   1.055
per leg, three times the lumped zone the camera saw:
   driver_u           45.6 K/W
   driver_v           45.6 K/W
   driver_w           45.6 K/W
   phase_u            45.6 K/W
   phase_v            45.6 K/W
   phase_w            45.6 K/W
fitted here against the same states:
   driver_u             86.5 K/W against the model  45.6
   driver_v             86.5 K/W against the model  45.6
   driver_w             86.5 K/W against the model  45.6
   mcu                  30.0 K/W against the model  22.5
   regulators            8.9 K/W against the model  15.0


The fit needs no least squares: `to_board = (T_zone - T_reference) / P_zone`, one division per node, with T from a camera against a dead patch of soldermask - not the NTC, which sits in the drivers' hot spot. A spreading resistance in the laminate is a few K/W; tens means either the power or the reference surface is wrong.

The order of the four states matters because each adds one power term to the one before, so the differences isolate a subsystem no single state can. Each was held 25 minutes: 3.7 times the board's 6.8 minute constant, 97 % of the way to equilibrium.

**The camera saw one bridge zone**, so it constrains the three legs together and not one of them. Per leg is three times the lumped 15.2 K/W and a third of the heat capacity, which leaves the three in parallel exactly what the camera measured while letting one leg alone rise three times as far and three times as fast - which is what the split is for, and why the nodes went per leg at CMD_PROTO MAJOR 2.

The NTC coupling is above 1 because it sits closer to the heat than the point its node stands for; capping it at 1.0 cost 5.6 K in the switching state before it was solved properly. The calibration behind all of it was taken **dry** - nothing on the phases, nothing through the hot swap - and at 100 A the shunt alone makes 35 W against the whole dry budget's 1.2 W.